In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append('/rhome/sawale/indus_traning/mlm-fine-tuning/mlm')

In [3]:
import os
import json
import random
import numpy as np
from transformers import AutoModelForMaskedLM, AutoTokenizer, pipeline, DataCollatorForLanguageModeling
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from dotenv import load_dotenv
load_dotenv()
from huggingface_hub import login
login(os.getenv("HUGGINGFACE_TOKEN"))

from preprocess_data import get_dataset


def set_seed(seed=42) -> None:
    """Set all seeds to make results reproducible (deterministic mode).
    When seed is a false-y value or not supplied, disables deterministic mode."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # Automatically select GPU if available
device

/nas/rhome/sawale/indus_traning/mlm-fine-tuning/.venv/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device(type='cuda')

# Load dataset

In [4]:
config_path = "../config.json"
with open(config_path, "r") as file:
    config = json.load(file)
data_src = "local"
n_rows = None

dataset = get_dataset(config["input"]["dataset"], data_src, n_rows)
print(dataset["test"].shape)

(12124, 13)


# Load the model, tokenizer, tokinze, collator

In [5]:
# Load model or dataset
new_model_name = "nasa-impact/indus-sde-v0.1"
old_model_name = "nasa-impact/nasa-smd-ibm-v0.1"

# Step 1: Load the model and tokenizer
model_names = {"old_model_name": "nasa-impact/nasa-smd-ibm-v0.1", "new_model_name": "nasa-impact/indus-sde-v0.1"}
models = {mn:AutoModelForMaskedLM.from_pretrained(hf_model_name).to(device) for mn, hf_model_name in model_names.items()}
tokenizers = {mn:AutoTokenizer.from_pretrained(hf_model_name) for mn, hf_model_name in model_names.items()}

# Function to tokenize the dataset
def tokenize_function(examples, tokenizer):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

# Tokenize datasets in advance
tokenized_datasets = {
    model_name: dataset.map(lambda examples: tokenize_function(examples, tokenizers[model_name]), batched=True, remove_columns=dataset["train"].column_names)
    for model_name in models.keys()
}

# Define data collator for MLM
data_collators = {
    model_name: DataCollatorForLanguageModeling(
        tokenizer=tokenizers[model_name],  # Use the correct tokenizer
        mlm=True,  # Enable Masked Language Modeling
        mlm_probability=0.15  # 15% of tokens will be masked
    ) 
    for model_name in models.keys()
}

losses = {mn:None for mn in model_names.keys()}


Map:   0%|                                                                                                                                                                                                                                                                                | 0/96992 [00:00<?, ? examples/s]


Map:   1%|██▋                                                                                                                                                                                                                                                                | 1000/96992 [00:00<00:34, 2815.95 examples/s]


Map:   2%|█████▎                                                                                                                                                                                                                                                             | 2000/96992 [00:00<00:29, 3195.85 examples/s]


Map:   3%|████████                                                                                                                                                                                                                                                           | 3000/96992 [00:00<00:31, 3020.55 examples/s]


Map:   4%|██████████▋                                                                                                                                                                                                                                                        | 4000/96992 [00:01<00:40, 2307.39 examples/s]

...


Map:  74%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▎                                                                  | 9000/12124 [00:05<00:01, 1717.50 examples/s]


Map:  82%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▊                                             | 10000/12124 [00:05<00:01, 2042.84 examples/s]


Map:  91%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████                        | 11000/12124 [00:06<00:00, 2352.34 examples/s]


Map:  99%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▎  | 12000/12124 [00:06<00:00, 2475.53 examples/s]


Map: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 12124/12124 [00:06<00:00, 1850.59 examples/s]

# Hepler Functions

In [6]:
# Function to compute loss
def compute_mlm_loss(split, model, data_collator, tokenized_dataset):
    dataloader = DataLoader(tokenized_dataset[split], batch_size=16, shuffle=False, collate_fn=data_collator)

    total_loss = 0
    total_samples = 0

    model.eval()  # Set model to evaluation mode
    with torch.no_grad():
        for batch in dataloader:
            batch = {k: v.to(model.device) for k, v in batch.items()}  # Move to correct device
            outputs = model(**batch)
            loss = outputs.loss
            total_loss += loss.item() * batch["input_ids"].shape[0]
            total_samples += batch["input_ids"].shape[0]

    return total_loss / total_samples

# Driver code

In [7]:
# Compute losses for both models
for model_name in models.keys():
    model = models[model_name]
    data_collator = data_collators[model_name]
    tokenized_dataset = tokenized_datasets[model_name]

    losses[model_name] = {
        "train_loss": compute_mlm_loss("train", model, data_collator, tokenized_dataset),
        "val_loss": compute_mlm_loss("validation", model, data_collator, tokenized_dataset),
        "test_loss": compute_mlm_loss("test", model, data_collator, tokenized_dataset)
    }

# Print results
for model_name, loss_values in losses.items():
    print(f"\nLoss for {model_name}:")
    print(f"  Train Loss: {loss_values['train_loss']:.4f}")
    print(f"  Validation Loss: {loss_values['val_loss']:.4f}")
    print(f"  Test Loss: {loss_values['test_loss']:.4f}")


Loss for old_model_name:
  Train Loss: 1.5537
  Validation Loss: 1.5543
  Test Loss: 1.5560

Loss for new_model_name:
  Train Loss: 0.5698
  Validation Loss: 0.5699
  Test Loss: 0.5703
